# Train the business recommendation model

This notebook trains a random forest regressor from `data/training_data.csv` and saves the fitted pipeline to `model/recommender.pkl`, which is loaded by `src/recommend.py`.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd().resolve()
# Find the project from either its root, the notebook folder, or a parent folder.
BASE_DIR = next(
    (parent for parent in (cwd, *cwd.parents)
     if (parent / 'data' / 'business_recommendation_dataset.csv').exists()),
    None,
)
if BASE_DIR is None:
    BASE_DIR = next(
        (parent / 'recommendationmodel' for parent in (cwd, *cwd.parents)
         if (parent / 'recommendationmodel' / 'data' / 'business_recommendation_dataset.csv').exists()),
        None,
    )
if BASE_DIR is None:
    raise FileNotFoundError(
        'Could not locate the recommendationmodel project folder. '
        'Open this notebook from inside recommendationmodel/notebook.'
    )

DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'model'
TRAINING_PATH = DATA_DIR / 'training_data.csv'
MODEL_PATH = MODEL_DIR / 'recommender.pkl'
if not TRAINING_PATH.exists():
    raise FileNotFoundError(
        f'Training data not found at {TRAINING_PATH}. '
        'Run create_dataset.ipynb from top to bottom first.'
    )
data = pd.read_csv(TRAINING_PATH)
features = ['investment', 'available_time', 'skill', 'land', 'experience', 'business']
target = 'score'
missing = set(features + [target]) - set(data.columns)
if missing:
    raise ValueError(f'Missing training columns: {sorted(missing)}')
print(f'Training on {len(data):,} examples')
data.head()

In [ ]:
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[('skill_and_business', OneHotEncoder(handle_unknown='ignore'), ['skill', 'business'])],
    remainder='passthrough',
)
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, predictions):.4f}')
print(f'R²:  {r2_score(y_test, predictions):.4f}')

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Saved trained model to {MODEL_PATH.resolve()}')

In [ ]:
# Preview a prediction using the same feature names expected by src/recommend.py
sample = X_test.head(3)
pd.DataFrame({'business': sample['business'], 'actual_score': y_test.loc[sample.index], 'predicted_score': model.predict(sample)})